In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import shapely

warnings.filterwarnings("ignore")

# Analysis Setup

This notebook covers the complete setup needed before starting an analysis with *PedPy*.
You will learn how to:

- Define the **walkable area** where pedestrians can move
- Set up **measurement areas** and **measurement lines** for your analysis
- **Import trajectory data** from various file formats

This guide uses a bottleneck experiment conducted at the University of Wuppertal in 2018 as a running example.

```{eval-rst}
.. figure:: demo-data/bottleneck/040_c_56_h-.png
    :width: 400px
    :align: center
```

The data for this experiment is available {download}`here <demo-data/bottleneck/040_c_56_h-.txt>`, which belongs to this [experimental series](https://doi.org/10.34735/ped.2018.1) and is part of the publication ["Crowds in front of bottlenecks at entrances from the perspective of physics and social psychology"](https://doi.org/10.1098/rsif.2019.0871).

## Measurement Setup



### Walkable area

In the beginning we will define the {class}`walkable area <geometry.WalkableArea>` in which the pedestrian can move. 
For the used bottleneck experiment was conducted in the following set up:


```{eval-rst}
.. figure:: demo-data/bottleneck/experimental_setup.png
    :width: 50 %
    :align: center
```

The run handled in this user guide had a bottleneck width of 0.5m and w=5.6m.

Below is the code for creating such a {class}`walkable area <geometry.WalkableArea>`:


In [ ]:
from pedpy import WalkableArea

walkable_area = WalkableArea(
    # complete area
    [
        (3.5, -2),
        (3.5, 8),
        (-3.5, 8),
        (-3.5, -2),
    ],
    obstacles=[
        # left barrier
        [
            (-0.7, -1.1),
            (-0.25, -1.1),
            (-0.25, -0.15),
            (-0.4, 0.0),
            (-2.8, 0.0),
            (-2.8, 6.7),
            (-3.05, 6.7),
            (-3.05, -0.3),
            (-0.7, -0.3),
            (-0.7, -1.0),
        ],
        # right barrier
        [
            (0.25, -1.1),
            (0.7, -1.1),
            (0.7, -0.3),
            (3.05, -0.3),
            (3.05, 6.7),
            (2.8, 6.7),
            (2.8, 0.0),
            (0.4, 0.0),
            (0.25, -0.15),
            (0.25, -1.1),
        ],
    ],
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_walkable_area

plot_walkable_area(walkable_area=walkable_area).set_aspect("equal")
plt.show()

### Measurement areas and measurement lines

After we defined where the pedestrians can move, we now need to define in which regions we want to analyze in more details. 
This regions can either be a specific line, an area, or the whole {class}`walkable area <geometry.WalkableArea>`.

In case of this bottleneck the most interesting area is a little bit in front of the bottleneck (here 0.5m) and the line at the beginning of the bottleneck.
The area is slightly in front of the bottleneck as here the highest density occur. 
In *PedPy* such areas are called {class}`~geometry.MeasurementArea` and the lines {class}`~geometry.MeasurementLine`.
Below you can see how to define these:


In [ ]:
from pedpy import MeasurementArea, MeasurementLine

measurement_area = MeasurementArea([(-0.4, 0.5), (0.4, 0.5), (0.4, 1.3), (-0.4, 1.3)])

measurement_line = MeasurementLine([(0.4, 0), (-0.4, 0)])

The corresponding measurement setup looks like:


In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

plot_measurement_setup(
    walkable_area=walkable_area,
    measurement_lines=[measurement_line],
    ml_width=2,
    measurement_areas=[measurement_area],
    ma_line_width=2,
    ma_alpha=0.2,
).set_aspect("equal")
plt.show()

## Importing pedestrian movement data

The pedestrian movement data in *PedPy* is called {class}`trajectory data<trajectory_data.TrajectoryData>`.

*PedPy* works with {class}`trajectory data<trajectory_data.TrajectoryData>` which can be created from an import function for specific data files alternatively from a {class}`~pandas.DataFrame` with the following columns:

- "id": unique numeric identifier for each person
- "frame": index of video frame where the positions were extracted
- "x", "y": position of the person (in meter)


### Loading from Pandas DataFrame

To construct the  {class}`trajectory data<trajectory_data.TrajectoryData>` from a {class}`~pandas.DataFrame` you also need to provide the frame rate at which the data was recorded.
If you have both the construction of the {class}`trajectory data<trajectory_data.TrajectoryData>` can be done with:


In [ ]:
from pedpy import TrajectoryData

data = pd.DataFrame(
    [[0, 1, 0, 0]],
    columns=["id", "frame", "x", "y"],
)
trajectory_data = TrajectoryData(data=data, frame_rate=25.0)

Alternatively, the data can be also loaded from any file format that is supported by *Pandas*, see the [documentation](https://pandas.pydata.org/pandas-docs/version/2.0/user_guide/io.html) for more details.


### Loading from text trajectory files

*Pedpy* can load trajectories, if they are stored as the {class}`trajectory data<trajectory_data.TrajectoryData>` provided in the [Jülich Data Archive](https://ped.fz-juelich.de/da/doku.php) directly.
If you have text files in same format, you can load them in the same way too:

- values are separated by any whitespace, e.g., space, tab
- file has at least 4 columns in the following order: "id", "frame", "x", "y"
- file may contain comment lines with `#` at in the beginning

For meaningful analysis (and loading of the trajectory file) you also need
- unit of the trajectory (m or cm)
- frame rate

For recent trajectory they are encoded in the header of the file, for older you may need to lead the documentation and provide the information in the loading process.

**Examples:**
With frame rate, but no unit
```
# description: UNI_CORR_500_01
# framerate: 25.00
#geometry: geometry.xml

# PersIDFrameXYZ
1 984.60121.89091.7600
1 994.53591.89761.7600
1 1004.44701.93041.7600
...
```

No header at all:
```
1 27 164.834 780.844 168.937
1 28 164.835 771.893 168.937
1 29 163.736 762.665 168.937
1 30 161.967 753.088 168.937
...
```

If your data is structured in a different way please take a look at the next section.
Since the data we want to analyze is from the data archive, we can directly load the {class}`trajectory data<trajectory_data.TrajectoryData>` with *PedPy*:


In [ ]:
from pedpy import TrajectoryUnit, load_trajectory

traj = load_trajectory(
    trajectory_file=pathlib.Path("demo-data/bottleneck/040_c_56_h-.txt"),
    default_unit=TrajectoryUnit.METER,  # needs to be provided as it not defined in the file
    # default_frame_rate=25., # can be ignored here as the frame rate is defined in the file
)

The loaded {class}`trajectory data<trajectory_data.TrajectoryData>` look like:


In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

plot_trajectories(traj=traj).set_aspect("equal")
plt.show()

### Loading from hdf5 trajectory files

For some experiments the [Jülich Data Archive](https://ped.fz-juelich.de/da/doku.php) also provides [HDF5](https://www.hdfgroup.org/HDF5) trajectory files, with a structure described [here](https://ped.fz-juelich.de/da/doku.php?id=info).
These data are from a different experiment, and are only used to demonstrate how to load HDF5 files, it can be downloaded {download}`here <demo-data/single_file/00_01a.h5>`.

To make the data usable for *PedPy* use:


In [ ]:
import pathlib

from pedpy import (
    TrajectoryData,
    load_trajectory_from_ped_data_archive_hdf5,
    load_walkable_area_from_ped_data_archive_hdf5,
)

h5_file = pathlib.Path("demo-data/single_file/00_01a.h5")

traj_h5 = load_trajectory_from_ped_data_archive_hdf5(trajectory_file=h5_file)
walkable_area_h5 = load_walkable_area_from_ped_data_archive_hdf5(trajectory_file=h5_file)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

plot_trajectories(traj=traj_h5, walkable_area=walkable_area_h5).set_aspect("equal")
plt.show()

### Loading from Viswalk trajectory files

It is also possible to load trajectory files from [Viswalk](https://www.ptvgroup.com/en/products/pedestrian-simulation-software-ptv-viswalk) directly into *PedPy*. 
The expected format is a CSV file with `;` as delimiter, and it should contain at least the following columns: `NO`, `SIMSEC`, `COORDCENTX`, `COORDCENTY`.
Comment lines may start with a `*` and will be ignored.

:::{important}
Currently only Viswalk trajectory files, which use the simulation time (`SIMSEC`) are supported.
:::


To make the data usable for *PedPy* use:


In [ ]:
import pathlib

from pedpy import (
    TrajectoryData,
    load_trajectory_from_viswalk,
)

viswalk_file = pathlib.Path("demo-data/viswalk/example.pp")

traj_viswalk = load_trajectory_from_viswalk(trajectory_file=viswalk_file)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

plot_trajectories(traj=traj_viswalk).set_aspect("equal")
plt.show()

### Loading from Vadere trajectory files

It is also possible to load trajectory files from [Vadere](https://www.vadere.org/) directly into *PedPy*. 
The expected format is a CSV file with space character as delimiter, and it should contain at least the following columns: `pedestrianId`, `simTime`, `startX`, `startY`.
Comment lines may start with a `#` and will be ignored.


To make the data usable for *PedPy* use:


In [ ]:
import pathlib

from pedpy import (
    TrajectoryData,
    load_trajectory_from_vadere,
    load_walkable_area_from_vadere_scenario,
)

vadere_traj_file = pathlib.Path("demo-data/vadere/bottleneck/vadere_postvis.traj")
vadere_scenario_file = pathlib.Path("demo-data/vadere/bottleneck/vadere_bottleneck.scenario")

traj_vadere = load_trajectory_from_vadere(trajectory_file=vadere_traj_file, frame_rate=24.0)
vadere_walkable_area = load_walkable_area_from_vadere_scenario(vadere_scenario_file=vadere_scenario_file, margin=1e-3)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

plot_trajectories(traj=traj_vadere, walkable_area=vadere_walkable_area).set_aspect("equal")
plt.show()

### Loading from Pathfinder trajectory files

[Pathfinder](https://www.thunderheadeng.com/pathfinder/) produces trajectory data in two formats: csv and json.

Both formats are supported by *PedPy* using the following calls:


In [ ]:
import pathlib

from pedpy import (
    load_trajectory_from_pathfinder_csv,
    load_trajectory_from_pathfinder_json,
)

traj_pathfinder_csv = load_trajectory_from_pathfinder_csv(
    trajectory_file=pathlib.Path("demo-data/pathfinder/pathfinder.csv")
)
traj_pathfinder_json = load_trajectory_from_pathfinder_json(
    trajectory_file=pathlib.Path("demo-data/pathfinder/pathfinder.json")
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
plot_trajectories(traj=traj_pathfinder_csv, axes=ax[0]).set_aspect("equal")
plot_trajectories(
    traj=traj_pathfinder_json,
    axes=ax[1],
).set_aspect("equal")
ax[0].set_title("Pathfinder CSV")
ax[1].set_title("Pathfinder JSON")
plt.show()

### Loading from Crowd:it trajectory files

[Crowd:it](https://www.accu-rate.de/en/software/crowdit/) generates a trajectory file and a geometry file. *PedPy* reads both out of the box as follows:


In [ ]:
import pathlib

from pedpy import load_trajectory_from_crowdit, load_walkable_area_from_crowdit

traj_crowdit = load_trajectory_from_crowdit(trajectory_file=pathlib.Path("demo-data/crowdit/crowdit.csv.gz"))
geometry_crowdit = load_walkable_area_from_crowdit(geometry_file=pathlib.Path("demo-data/crowdit/crowdit.floor"))

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

plot_trajectories(traj=traj_crowdit, walkable_area=geometry_crowdit).set_aspect("equal")
plt.show()

### Loading from JuPedSim trajectory files

[JuPedSim](https://jupedsim.org) produces trajectory data in SQLite format containing the trajectories as well as the geometry data in WKT format. 
*PedPy* supports both natively:


In [ ]:
from pedpy import (
    load_trajectory_from_jupedsim_sqlite,
    load_walkable_area_from_jupedsim_sqlite,
)

traj_jupedsim = load_trajectory_from_jupedsim_sqlite(
    trajectory_file=pathlib.Path("demo-data/jupedsim/trajectories.sqlite")
)
geometry_jupedsim = load_walkable_area_from_jupedsim_sqlite(
    trajectory_file=pathlib.Path("demo-data/jupedsim/trajectories.sqlite")
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_trajectories

plot_trajectories(traj=traj_jupedsim, walkable_area=geometry_jupedsim).set_aspect("equal")
plt.show()